# Parquet → Neo4j AuraDB importer

This notebook reads the Microsoft Fabric graph sample tables from a **generic base path**,
discovers the tables, builds an import plan, and writes the graph to **Neo4j AuraDB**.

It supports:

- **LDBC social-network sample** with fully automatic node/edge discovery
- **AdventureWorks sample** with auto-discovered tables plus a built-in relationship map
- direct reads from **Delta folders** under a single base path
- batch writes to Aura using the Neo4j Python driver
- a **dry-run** mode so you can inspect the plan before loading data

> Expected layout:
> a root folder containing one subfolder per table, where each table folder is a Delta table.

In [63]:
# If needed in Fabric / Spark, install once per session.
# Comment this out if the packages are already available.
# For local Spark + Delta support, uncomment the next line too.
# %pip install -q pyspark delta-spark
# %pip install -q neo4j python-dotenv

In [64]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("local-test").getOrCreate()

print("Spark =", spark.version)
print(
    "JVM java.version =",
    spark.sparkContext._jvm.java.lang.System.getProperty("java.version"),
)
print(
    "JVM java.home =",
    spark.sparkContext._jvm.java.lang.System.getProperty("java.home"),
)

Spark = 4.1.1
JVM java.version = 21.0.10
JVM java.home = /opt/homebrew/Cellar/openjdk@21/21.0.10/libexec/openjdk.jdk/Contents/Home


In [ ]:
# =========================
# Configuration
# =========================

import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
load_dotenv(Path(".env"), override=False)
load_dotenv(Path("graph-db") / ".env", override=False)

def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Set {name} in your .env file")
    return value

# Point this to the extracted dataset root folder.
# Examples:
#   Files/graph/adventureworks_docs_sample
#   Files/graph/ldbc_snb_docs_sample
BASE_PATH = "sample_data/ldbc_snb_docs_sample"

# Make the bundled sample data work whether the notebook is launched from
# the repo root or from the graph-db folder.
if not Path(BASE_PATH).exists() and Path("graph-db", BASE_PATH).exists():
    BASE_PATH = str(Path("graph-db") / BASE_PATH)

# Aura connection loaded from .env
NEO4J_URI = require_env("NEO4J_URI")
NEO4J_USERNAME = require_env("NEO4J_USERNAME")
NEO4J_PASSWORD = require_env("NEO4J_PASSWORD")

# Auto / ldbc / adventureworks
DATASET_HINT = "auto"

# Auto uses Parquet files for local Delta folders and Delta reader for cloud/URI paths.
# Use "delta" in Fabric/Databricks if you want to force the Delta reader.
LOCAL_DELTA_READ_MODE = "auto"  # auto / parquet / delta

# Safety controls
DRY_RUN = True
BATCH_SIZE = 1000
CREATE_CONSTRAINTS = True
IMPORT_VENDORPRODUCT_AS_NODE = False  # normally False for AdventureWorks
USE_UPPERCASE_REL_TYPES = True
STOP_ON_FIRST_ERROR = True

print(f"BASE_PATH={BASE_PATH}")
print(f"DATASET_HINT={DATASET_HINT}")
print("NEO4J_URI loaded from environment")
print(f"DRY_RUN={DRY_RUN}, BATCH_SIZE={BATCH_SIZE}")

BASE_PATH=sample_data/ldbc_snb_docs_sample
DATASET_HINT=auto
NEO4J_URI loaded from environment
DRY_RUN=False, BATCH_SIZE=1000


In [66]:
# =========================
# Imports
# =========================

import base64
import json
import re
from datetime import date, datetime, time
from decimal import Decimal
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional

from neo4j import GraphDatabase
from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F

def get_or_create_spark():
    existing = globals().get("spark")
    if existing is not None:
        return existing

    builder = SparkSession.builder.appName("parquet-to-auradb")

    try:
        from delta import configure_spark_with_delta_pip
    except ImportError:
        return builder.getOrCreate()

    builder = (
        builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()

spark = get_or_create_spark()
print(f"Using Spark {spark.version}")

Using Spark 4.1.1


In [67]:
print("Spark:", spark.version)
print("Java:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))

Spark: 4.1.1
Java: 21.0.10


In [68]:
# =========================
# Filesystem helpers
# =========================

def _is_uri_path(path: str) -> bool:
    return re.match(r"^[A-Za-z][A-Za-z0-9+.-]*:", path) is not None

def _local_path(path: str) -> Path:
    return Path(path).expanduser()

def _hadoop_fs():
    jvm = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    return jvm.org.apache.hadoop.fs.FileSystem.get(hconf), jvm.org.apache.hadoop.fs.Path

def normalize_path(path: str) -> str:
    path = str(path).strip()
    if "://" in path:
        scheme, rest = path.split("://", 1)
        return f"{scheme}://{re.sub(r'/+', '/', rest).rstrip('/')}"
    return re.sub(r"/+", "/", path).rstrip("/")

def path_exists(path: str) -> bool:
    path = normalize_path(path)
    if not _is_uri_path(path) and _local_path(path).exists():
        return True

    try:
        fs, HadoopPath = _hadoop_fs()
        return fs.exists(HadoopPath(path))
    except Exception:
        return False

def list_status(path: str):
    fs, HadoopPath = _hadoop_fs()
    return fs.listStatus(HadoopPath(path))

def list_child_dirs(path: str) -> List[str]:
    path = normalize_path(path)
    local = _local_path(path)
    if not _is_uri_path(path) and local.exists():
        return sorted(str(p) for p in local.iterdir() if p.is_dir())

    if not path_exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")
    return sorted(
        [str(s.getPath().toString()) for s in list_status(path) if s.isDirectory()]
    )

def basename(path: str) -> str:
    return normalize_path(path).split("/")[-1]

def is_delta_table(path: str) -> bool:
    path = normalize_path(path)
    delta_log = path + "/_delta_log"
    if not _is_uri_path(path) and _local_path(delta_log).exists():
        return True

    try:
        fs, HadoopPath = _hadoop_fs()
        return fs.exists(HadoopPath(delta_log))
    except Exception:
        return False

def local_parquet_files(path: str) -> List[str]:
    local = _local_path(normalize_path(path))
    if _is_uri_path(path) or not local.exists():
        return []
    return sorted(str(p.resolve()) for p in local.glob("*.parquet"))

def read_delta_table(path: str):
    path = normalize_path(path)
    mode = LOCAL_DELTA_READ_MODE.lower()
    parquet_files = local_parquet_files(path)

    if parquet_files and mode in {"auto", "parquet"}:
        return spark.read.parquet(*parquet_files)

    try:
        return spark.read.format("delta").load(path)
    except Exception as exc:
        if parquet_files and mode == "delta":
            raise RuntimeError(
                "Spark could not read this local Delta table with the Delta reader. "
                "Set LOCAL_DELTA_READ_MODE='auto' or 'parquet' for the bundled local sample data, "
                "or restart the kernel after installing compatible pyspark and delta-spark packages."
            ) from exc
        raise

def discover_delta_tables(base_path: str) -> List[Dict]:
    base_path = normalize_path(base_path)
    dirs = list_child_dirs(base_path)
    tables = []

    for child in dirs:
        name = basename(child)
        if is_delta_table(child):
            df = read_delta_table(child)
            tables.append(
                {
                    "table_name": name,
                    "path": child,
                    "columns": df.columns,
                    "row_count": None,
                }
            )
    return tables

tables = discover_delta_tables(BASE_PATH)
print(f"Discovered {len(tables)} Delta tables under {BASE_PATH}")
for t in tables[:20]:
    print(f"- {t['table_name']}: {len(t['columns'])} columns")

Discovered 36 Delta tables under sample_data/ldbc_snb_docs_sample
- ldbc_snb_edge_City_isPartOf_Country: 4 columns
- ldbc_snb_edge_Comment_hasCreator_Person: 4 columns
- ldbc_snb_edge_Comment_hasTag_Tag: 4 columns
- ldbc_snb_edge_Comment_isLocatedIn_Country: 4 columns
- ldbc_snb_edge_Comment_replyOf_Comment: 4 columns
- ldbc_snb_edge_Comment_replyOf_Post: 4 columns
- ldbc_snb_edge_Company_isLocatedIn_Country: 4 columns
- ldbc_snb_edge_Country_isPartOf_Continent: 4 columns
- ldbc_snb_edge_Forum_containerOf_Post: 4 columns
- ldbc_snb_edge_Forum_hasMember_Person: 5 columns
- ldbc_snb_edge_Forum_hasModerator_Person: 4 columns
- ldbc_snb_edge_Forum_hasTag_Tag: 4 columns
- ldbc_snb_edge_Person_hasInterest_Tag: 4 columns
- ldbc_snb_edge_Person_isLocatedIn_City: 4 columns
- ldbc_snb_edge_Person_knows_Person: 5 columns
- ldbc_snb_edge_Person_likes_Comment: 5 columns
- ldbc_snb_edge_Person_likes_Post: 5 columns
- ldbc_snb_edge_Person_studyAt_University: 5 columns
- ldbc_snb_edge_Person_workAt_Co

In [69]:
# =========================
# Naming helpers
# =========================

def snake_to_pascal(s: str) -> str:
    return "".join(p.capitalize() for p in re.split(r"[_\W]+", s) if p)

def sanitize_identifier(value: str) -> str:
    return value.replace("`", "")

def normalize_label(raw: str) -> str:
    raw = raw.strip("_")
    special = {
        "tagclass": "TagClass",
        "tag_class": "TagClass",
        "place": "Place",
        "city": "City",
        "country": "Country",
        "continent": "Continent",
        "organization": "Organization",
        "university": "University",
        "company": "Company",
        "person": "Person",
        "forum": "Forum",
        "message": "Message",
        "post": "Post",
        "comment": "Comment",
        "tag": "Tag",
        "customer": "Customer",
        "employee": "Employee",
        "order": "Order",
        "product": "Product",
        "productcategory": "ProductCategory",
        "productsubcategory": "ProductSubcategory",
        "vendor": "Vendor",
        "vendorproduct": "VendorProduct",
    }
    key = raw.lower()
    return special.get(key, snake_to_pascal(raw))

def normalize_rel_type(raw: str) -> str:
    raw = raw.strip("_")
    if USE_UPPERCASE_REL_TYPES:
        return re.sub(r"[^A-Za-z0-9_]", "_", raw).upper()
    return snake_to_pascal(raw)

def detect_dataset_family(table_names: List[str]) -> str:
    lowered = [t.lower() for t in table_names]
    if any(t.startswith("ldbc_snb_node_") or t.startswith("ldbc_snb_edge_") for t in lowered):
        return "ldbc"
    if any(t.startswith("adventureworks_") for t in lowered):
        return "adventureworks"
    return "unknown"

dataset_family = DATASET_HINT.lower()
if dataset_family == "auto":
    dataset_family = detect_dataset_family([t["table_name"] for t in tables])

print("Detected dataset family:", dataset_family)

Detected dataset family: ldbc


In [70]:
# =========================
# Value conversion helpers
# =========================

def sanitize_value(v):
    if v is None:
        return None
    if isinstance(v, (str, int, float, bool)):
        return v
    if isinstance(v, Decimal):
        if v == v.to_integral_value():
            return int(v)
        return float(v)
    if isinstance(v, (date, datetime, time)):
        return v.isoformat()
    if isinstance(v, bytes):
        return base64.b64encode(v).decode("utf-8")
    if isinstance(v, Row):
        return json.dumps(v.asDict(recursive=True), default=str, ensure_ascii=False)
    if isinstance(v, dict):
        return json.dumps(v, default=str, ensure_ascii=False)
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            s = sanitize_value(item)
            if isinstance(s, (dict, list, tuple)):
                out.append(json.dumps(s, default=str, ensure_ascii=False))
            else:
                out.append(s)
        return out
    return str(v)

def row_to_dict(row) -> Dict:
    if isinstance(row, Row):
        raw = row.asDict(recursive=True)
    else:
        raw = dict(row)
    return {k: sanitize_value(v) for k, v in raw.items()}

def batched(iterable: Iterable, size: int = 1000) -> Iterator[List]:
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch

In [71]:
# =========================
# Table readers
# =========================

table_index = {t["table_name"]: t for t in tables}

def read_table_df(table_name: str):
    info = table_index[table_name]
    return read_delta_table(info["path"])

def preview_tables(limit: int = 20):
    rows = []
    for t in tables[:limit]:
        rows.append((t["table_name"], t["path"], len(t["columns"]), ", ".join(t["columns"][:8])))
    display(spark.createDataFrame(rows, ["table_name", "path", "num_columns", "first_columns"]))

preview_tables()

DataFrame[table_name: string, path: string, num_columns: bigint, first_columns: string]

In [72]:
# =========================
# Plan builders
# =========================

ADVENTUREWORKS_NODE_MAP = {
    "adventureworks_customers": {"label": "Customer", "key_col": "CustomerID_K"},
    "adventureworks_employees": {"label": "Employee", "key_col": "EmployeeID_K"},
    "adventureworks_orders": {"label": "Order", "key_col": "SalesOrderDetailID_K"},
    "adventureworks_products": {"label": "Product", "key_col": "ProductID_K"},
    "adventureworks_productcategories": {"label": "ProductCategory", "key_col": "CategoryID_K"},
    "adventureworks_productsubcategories": {"label": "ProductSubcategory", "key_col": "SubcategoryID_K"},
    "adventureworks_vendors": {"label": "Vendor", "key_col": "VendorID_K"},
}

ADVENTUREWORKS_EDGE_MAP = [
    {
        "table_name": "adventureworks_orders",
        "src_label": "Employee",
        "src_col": "EmployeeID_FK",
        "dst_label": "Order",
        "dst_col": "SalesOrderDetailID_K",
        "rel_type": "SOLD",
    },
    {
        "table_name": "adventureworks_orders",
        "src_label": "Customer",
        "src_col": "CustomerID_FK",
        "dst_label": "Order",
        "dst_col": "SalesOrderDetailID_K",
        "rel_type": "PURCHASED",
    },
    {
        "table_name": "adventureworks_orders",
        "src_label": "Order",
        "src_col": "SalesOrderDetailID_K",
        "dst_label": "Product",
        "dst_col": "ProductID_FK",
        "rel_type": "CONTAINS",
    },
    {
        "table_name": "adventureworks_products",
        "src_label": "Product",
        "src_col": "ProductID_K",
        "dst_label": "ProductSubcategory",
        "dst_col": "SubcategoryID_FK",
        "rel_type": "IS_OF_TYPE",
    },
    {
        "table_name": "adventureworks_productsubcategories",
        "src_label": "ProductSubcategory",
        "src_col": "SubcategoryID_K",
        "dst_label": "ProductCategory",
        "dst_col": "CategoryID_FK",
        "rel_type": "BELONGS_TO",
    },
    {
        "table_name": "adventureworks_vendorproduct",
        "src_label": "Vendor",
        "src_col": "VendorID_FK",
        "dst_label": "Product",
        "dst_col": "ProductID_FK",
        "rel_type": "PRODUCES",
    },
]

def infer_ldbc_node_spec(table_name: str):
    prefix = "ldbc_snb_node_"
    if not table_name.lower().startswith(prefix):
        return None
    raw = table_name[len(prefix):]
    return {"table_name": table_name, "label": normalize_label(raw), "key_col": "id"}

def infer_ldbc_edge_spec(table_name: str, columns: List[str]):
    prefix = "ldbc_snb_edge_"
    if not table_name.lower().startswith(prefix):
        return None

    src_candidates = [c for c in columns if re.match(r"^src_.+_id$", c)]
    dst_candidates = [c for c in columns if re.match(r"^dst_.+_id$", c)]
    if not src_candidates or not dst_candidates:
        return None

    src_col = sorted(src_candidates)[0]
    dst_col = sorted(dst_candidates)[0]

    src_raw = re.sub(r"^src_(.+)_id$", r"\1", src_col)
    dst_raw = re.sub(r"^dst_(.+)_id$", r"\1", dst_col)
    src_label = normalize_label(src_raw)
    dst_label = normalize_label(dst_raw)

    remainder = table_name[len(prefix):]
    rel_raw = remainder
    src_prefix = f"{src_raw}_"
    dst_suffix = f"_{dst_raw}"

    if rel_raw.startswith(src_prefix):
        rel_raw = rel_raw[len(src_prefix):]
    if rel_raw.endswith(dst_suffix):
        rel_raw = rel_raw[: -len(dst_suffix)]
    if not rel_raw:
        rel_raw = "related_to"

    return {
        "table_name": table_name,
        "src_label": src_label,
        "src_col": src_col,
        "dst_label": dst_label,
        "dst_col": dst_col,
        "rel_type": normalize_rel_type(rel_raw),
    }

def build_import_plan(tables: List[Dict], dataset_family: str):
    nodes, edges = [], []

    if dataset_family == "ldbc":
        for t in tables:
            node_spec = infer_ldbc_node_spec(t["table_name"])
            if node_spec:
                nodes.append(node_spec)
                continue
            edge_spec = infer_ldbc_edge_spec(t["table_name"], t["columns"])
            if edge_spec:
                edges.append(edge_spec)

    elif dataset_family == "adventureworks":
        available = {t["table_name"] for t in tables}

        for table_name, spec in ADVENTUREWORKS_NODE_MAP.items():
            if table_name in available:
                nodes.append({"table_name": table_name, **spec})

        if IMPORT_VENDORPRODUCT_AS_NODE and "adventureworks_vendorproduct" in available:
            nodes.append(
                {
                    "table_name": "adventureworks_vendorproduct",
                    "label": "VendorProduct",
                    "key_col": "ProductID_FK",
                }
            )

        for spec in ADVENTUREWORKS_EDGE_MAP:
            if spec["table_name"] in available:
                edges.append(spec.copy())

    else:
        raise ValueError(
            f"Unsupported dataset family '{dataset_family}'. "
            "Use 'ldbc', 'adventureworks', or add your own plan builder."
        )

    return {"nodes": nodes, "edges": edges}

plan = build_import_plan(tables, dataset_family)
print(f"Node specs: {len(plan['nodes'])}")
print(f"Edge specs: {len(plan['edges'])}")
plan

Node specs: 11
Edge specs: 25


{'nodes': [{'table_name': 'ldbc_snb_node_City',
   'label': 'City',
   'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Comment', 'label': 'Comment', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Company', 'label': 'Company', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Continent',
   'label': 'Continent',
   'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Country', 'label': 'Country', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Forum', 'label': 'Forum', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Person', 'label': 'Person', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Post', 'label': 'Post', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_Tag', 'label': 'Tag', 'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_TagClass',
   'label': 'TagClass',
   'key_col': 'id'},
  {'table_name': 'ldbc_snb_node_University',
   'label': 'University',
   'key_col': 'id'}],
 'edges': [{'table_name': 'ldbc_snb_edge_City_isPartOf_Country',
   'src_label': 'Place',
   'src_co

In [73]:
# =========================
# Validate the plan
# =========================

def validate_plan(plan, table_index):
    issues = []

    for spec in plan["nodes"]:
        if spec["table_name"] not in table_index:
            issues.append(f"Missing node table: {spec['table_name']}")
            continue
        cols = set(table_index[spec["table_name"]]["columns"])
        if spec["key_col"] not in cols:
            issues.append(f"Node key column not found: {spec['table_name']}.{spec['key_col']}")

    for spec in plan["edges"]:
        if spec["table_name"] not in table_index:
            issues.append(f"Missing edge table: {spec['table_name']}")
            continue
        cols = set(table_index[spec["table_name"]]["columns"])
        if spec["src_col"] not in cols:
            issues.append(f"Edge source column not found: {spec['table_name']}.{spec['src_col']}")
        if spec["dst_col"] not in cols:
            issues.append(f"Edge target column not found: {spec['table_name']}.{spec['dst_col']}")

    return issues

issues = validate_plan(plan, table_index)
if issues:
    print("Validation issues:")
    for i in issues:
        print("-", i)
    raise ValueError("Plan validation failed. Fix the missing columns/tables above.")
else:
    print("Plan validation passed.")

Plan validation passed.


In [74]:
# Show the node and edge plan
node_rows = [(s["table_name"], s["label"], s["key_col"]) for s in plan["nodes"]]
edge_rows = [
    (s["table_name"], s["src_label"], s["src_col"], s["rel_type"], s["dst_label"], s["dst_col"])
    for s in plan["edges"]
]

if node_rows:
    print("Nodes")
    display(spark.createDataFrame(node_rows, ["table_name", "label", "key_col"]))

if edge_rows:
    print("Edges")
    display(spark.createDataFrame(edge_rows, ["table_name", "src_label", "src_col", "rel_type", "dst_label", "dst_col"]))

Nodes


DataFrame[table_name: string, label: string, key_col: string]

Edges


DataFrame[table_name: string, src_label: string, src_col: string, rel_type: string, dst_label: string, dst_col: string]

In [75]:
# =========================
# Neo4j driver helpers
# =========================

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_write_batches(cypher: str, rows: Iterable[Dict], batch_size: int = 1000, description: str = ""):
    total = 0
    with driver.session() as session:
        for batch in batched(rows, batch_size):
            session.run(cypher, rows=batch).consume()
            total += len(batch)
            if total % (batch_size * 10) == 0:
                print(f"{description} -> {total:,} rows written")
    return total

def create_constraint(label: str):
    label = sanitize_identifier(label)
    cypher = f"CREATE CONSTRAINT {label.lower()}_id IF NOT EXISTS FOR (n:`{label}`) REQUIRE n.id IS UNIQUE"
    with driver.session() as session:
        session.run(cypher).consume()

def close_driver():
    driver.close()

In [76]:
# =========================
# Import functions
# =========================

def node_rows_iter(df, key_col: str):
    select_cols = [c for c in df.columns if c != key_col]
    projected = df.select(F.col(key_col).alias("id"), *[F.col(c) for c in select_cols])
    for row in projected.toLocalIterator():
        yield row_to_dict(row)

def edge_rows_iter(df, src_col: str, dst_col: str):
    projected = df.select(
        F.col(src_col).alias("_src"),
        F.col(dst_col).alias("_dst"),
        *[F.col(c) for c in df.columns if c not in {src_col, dst_col}]
    )
    for row in projected.toLocalIterator():
        record = row_to_dict(row)
        if record.get("_src") is None or record.get("_dst") is None:
            continue
        yield record

def import_nodes(table_name: str, label: str, key_col: str, batch_size: int = 1000):
    df = read_table_df(table_name)

    cypher = f"""
    UNWIND $rows AS row
    MERGE (n:`{sanitize_identifier(label)}` {{id: row.id}})
    SET n += row
    """

    total = run_write_batches(
        cypher,
        node_rows_iter(df, key_col=key_col),
        batch_size=batch_size,
        description=f"nodes:{label}",
    )
    print(f"Imported {total:,} nodes for label {label}")

def import_edges(
    table_name: str,
    src_label: str,
    src_col: str,
    dst_label: str,
    dst_col: str,
    rel_type: str,
    batch_size: int = 1000,
):
    df = read_table_df(table_name)

    cypher = f"""
    UNWIND $rows AS row
    MATCH (s:`{sanitize_identifier(src_label)}` {{id: row._src}})
    MATCH (t:`{sanitize_identifier(dst_label)}` {{id: row._dst}})
    MERGE (s)-[r:`{sanitize_identifier(rel_type)}`]->(t)
    SET r += row
    REMOVE r._src, r._dst
    """

    total = run_write_batches(
        cypher,
        edge_rows_iter(df, src_col=src_col, dst_col=dst_col),
        batch_size=batch_size,
        description=f"edges:{src_label}-[{rel_type}]->{dst_label}",
    )
    print(f"Imported {total:,} relationships for type {rel_type}")

In [77]:
# =========================
# Optional row counts before import
# =========================

COUNT_TABLES = True

if COUNT_TABLES:
    counts = []
    for t in tables:
        cnt = read_table_df(t["table_name"]).count()
        counts.append((t["table_name"], cnt))
    display(spark.createDataFrame(counts, ["table_name", "row_count"]))
else:
    print("Skipping row counts. Set COUNT_TABLES=True if you want exact counts before import.")

DataFrame[table_name: string, row_count: bigint]

In [78]:
# =========================
# Dry run summary
# =========================

print("Dry run summary")
print("---------------")
print("Dataset family:", dataset_family)
print("Base path:", BASE_PATH)
print("Node labels:", [n["label"] for n in plan["nodes"]])
print("Relationship types:", [e["rel_type"] for e in plan["edges"]])

if DRY_RUN:
    print("\nDRY_RUN=True, so no writes will be performed.")
else:
    print("\nDRY_RUN=False, writes will be performed.")

Dry run summary
---------------
Dataset family: ldbc
Base path: sample_data/ldbc_snb_docs_sample
Node labels: ['City', 'Comment', 'Company', 'Continent', 'Country', 'Forum', 'Person', 'Post', 'Tag', 'TagClass', 'University']
Relationship types: ['CITY_ISPARTOF_COUNTRY', 'HASCREATOR', 'HASTAG', 'ISLOCATEDIN_COUNTRY', 'REPLYOF', 'REPLYOF', 'COMPANY_ISLOCATEDIN_COUNTRY', 'COUNTRY_ISPARTOF_CONTINENT', 'CONTAINEROF', 'HASMEMBER', 'HASMODERATOR', 'HASTAG', 'HASINTEREST', 'ISLOCATEDIN_CITY', 'KNOWS', 'LIKES', 'LIKES', 'STUDYAT_UNIVERSITY', 'WORKAT_COMPANY', 'HASCREATOR', 'HASTAG', 'ISLOCATEDIN_COUNTRY', 'ISSUBCLASSOF', 'HASTYPE', 'UNIVERSITY_ISLOCATEDIN_CITY']

DRY_RUN=False, writes will be performed.


In [79]:
# =========================
# Execute import
# =========================

if DRY_RUN:
    print("Dry run only. Flip DRY_RUN=False and rerun this cell to write to Aura.")
else:
    try:
        if CREATE_CONSTRAINTS:
            labels = sorted({spec["label"] for spec in plan["nodes"]})
            for label in labels:
                print(f"Creating constraint for :{label}(id)")
                create_constraint(label)

        for spec in plan["nodes"]:
            print(f"Importing node table {spec['table_name']} -> :{spec['label']}")
            import_nodes(
                table_name=spec["table_name"],
                label=spec["label"],
                key_col=spec["key_col"],
                batch_size=BATCH_SIZE,
            )

        for spec in plan["edges"]:
            print(
                f"Importing edge table {spec['table_name']} -> "
                f"(:{spec['src_label']})-[:{spec['rel_type']}]->(:{spec['dst_label']})"
            )
            import_edges(
                table_name=spec["table_name"],
                src_label=spec["src_label"],
                src_col=spec["src_col"],
                dst_label=spec["dst_label"],
                dst_col=spec["dst_col"],
                rel_type=spec["rel_type"],
                batch_size=BATCH_SIZE,
            )

        print("Import complete.")
    except Exception as e:
        print("Import failed:", e)
        if STOP_ON_FIRST_ERROR:
            raise

Creating constraint for :City(id)
Creating constraint for :Comment(id)
Creating constraint for :Company(id)
Creating constraint for :Continent(id)
Creating constraint for :Country(id)
Creating constraint for :Forum(id)
Creating constraint for :Person(id)
Creating constraint for :Post(id)
Creating constraint for :Tag(id)
Creating constraint for :TagClass(id)
Creating constraint for :University(id)
Importing node table ldbc_snb_node_City -> :City
Imported 1,343 nodes for label City
Importing node table ldbc_snb_node_Comment -> :Comment
nodes:Comment -> 10,000 rows written
nodes:Comment -> 20,000 rows written
nodes:Comment -> 30,000 rows written
nodes:Comment -> 40,000 rows written
nodes:Comment -> 50,000 rows written
nodes:Comment -> 60,000 rows written
nodes:Comment -> 70,000 rows written
nodes:Comment -> 80,000 rows written
nodes:Comment -> 90,000 rows written
nodes:Comment -> 100,000 rows written
nodes:Comment -> 110,000 rows written
nodes:Comment -> 120,000 rows written
nodes:Comment

ClientError: {neo4j_code: Neo.ClientError.Transaction.TransactionHookFailed} {message: You have exceeded the logical size limit of 200000 nodes in your database (attempt to add 1000 nodes would reach 200356 nodes). Please consider upgrading to the next tier.} {gql_status: 50N00} {gql_status_description: error: general processing exception - internal error. Internal exception raised TransactionEventListeners: You have exceeded the logical size limit of 200000 nodes in your database (attempt to add 1000 nodes would reach 200356 nodes). Please consider upgrading to the next tier.}

In [ ]:
# =========================
# Post-load sanity checks
# =========================

SANITY_CHECKS = True

if SANITY_CHECKS and not DRY_RUN:
    queries = [
        "MATCH (n) RETURN count(n) AS node_count",
        "MATCH ()-[r]->() RETURN count(r) AS rel_count",
        "CALL db.labels()",
        "CALL db.relationshipTypes()",
    ]
    with driver.session() as session:
        for q in queries:
            print("\nQuery:", q)
            result = session.run(q)
            for record in result:
                print(record)
else:
    print("Skipping sanity checks. Set SANITY_CHECKS=True after a real import if needed.")

Skipping sanity checks. Set SANITY_CHECKS=True after a real import if needed.


In [ ]:
# Close the Neo4j driver when you're done.
# close_driver()